The idea behind this notebook is to check the real results of the model, and also check how it compares with the bets

In [1]:
import pandas as pd

from validators import add_country_code

df_results = pd.read_csv('first_round_results.csv')


In [2]:
df_results = add_country_code(df=df_results, target_column="team_1", output_column="team_1")
df_results = add_country_code(df=df_results, target_column="team_2", output_column="team_2")
df_results = add_country_code(df=df_results, target_column="winner", output_column="winner", replace_none="Draw")
df_results

,Unnamed: 0,team_1,team_2,goals_1,goals_2,winner
0,0,MX,ZA,2,0,MX
1,1,KR,CZ,2,1,KR
2,2,CA,BA,1,1,Draw
3,3,QA,CH,1,1,Draw
4,4,BR,MA,1,1,Draw
5,5,HT,SQ,0,1,SQ
6,6,US,PY,4,1,US
7,7,AU,TR,2,0,AU
8,8,DE,CW,7,1,DE
9,9,CI,EC,1,0,CI


In [3]:
local_predictions = pd.read_csv('first_round_predictions.csv')
local_predictions = add_country_code(df=local_predictions, target_column="team_1", output_column="team_1")
local_predictions = add_country_code(df=local_predictions, target_column="team_2", output_column="team_2")
local_predictions = add_country_code(df=local_predictions, target_column="winner", output_column="winner", replace_none="Draw")
def add_draw(row):
    confidence_threshold = 1 # actually one goal
    confidence = abs(row['goals_1'] - row['goals_2'])
    if confidence < confidence_threshold:
        row['winner'] = 'Draw'
    return row

local_predictions = local_predictions.apply(add_draw, axis=1)
local_predictions

,Unnamed: 0,team_1,team_2,goals_1,goals_2,winner,location
0,0,MX,ZA,1.992725,3.649461e-01,MX,MX
1,1,KR,CZ,2.018553,1.521301e+00,Draw,MX
2,2,CA,BA,1.552012,1.458836e+00,Draw,CA
3,3,QA,CH,0.418108,5.799674e+00,CH,US
4,4,BR,MA,1.797937,1.221405e+00,Draw,US
5,5,HT,SQ,2.389982,4.907077e+00,SQ,US
6,6,US,PY,4.039485,1.005360e+00,US,US
7,7,AU,TR,1.036107,2.631848e+00,TR,CA
8,8,DE,CW,11.508186,4.392235e-01,DE,US
9,9,CI,EC,2.685137,2.314583e+00,Draw,US


In [4]:
# how much difference on the winner?
correct_winners = 0

for row_in_real in df_results.itertuples():
    print(f"{row_in_real.team_1} vs {row_in_real.team_2}: {row_in_real.goals_1} - {row_in_real.goals_2} = {row_in_real.winner}")
    for row_in_prediction in local_predictions.itertuples():
        if row_in_real.team_1 == row_in_prediction.team_1 and row_in_real.team_2 == row_in_prediction.team_2 or row_in_real.team_1 == row_in_prediction.team_2 and row_in_real.team_2 == row_in_prediction.team_1:
            print(f"{row_in_prediction.team_1} vs {row_in_prediction.team_2}: {row_in_prediction.goals_1:.1f} - {row_in_prediction.goals_2:.1f} = {row_in_prediction.winner}")
            if row_in_real.winner == row_in_prediction.winner:
                correct_winners += 1
                print("Correct!")
            else:
                print("Incorrect!")



MX vs ZA: 2 - 0 = MX
MX vs ZA: 2.0 - 0.4 = MX
Correct!
KR vs CZ: 2 - 1 = KR
KR vs CZ: 2.0 - 1.5 = Draw
Incorrect!
CA vs BA: 1 - 1 = Draw
CA vs BA: 1.6 - 1.5 = Draw
Correct!
QA vs CH: 1 - 1 = Draw
QA vs CH: 0.4 - 5.8 = CH
Incorrect!
BR vs MA: 1 - 1 = Draw
BR vs MA: 1.8 - 1.2 = Draw
Correct!
HT vs SQ: 0 - 1 = SQ
HT vs SQ: 2.4 - 4.9 = SQ
Correct!
US vs PY: 4 - 1 = US
US vs PY: 4.0 - 1.0 = US
Correct!
AU vs TR: 2 - 0 = AU
AU vs TR: 1.0 - 2.6 = TR
Incorrect!
DE vs CW: 7 - 1 = DE
DE vs CW: 11.5 - 0.4 = DE
Correct!
CI vs EC: 1 - 0 = CI
CI vs EC: 2.7 - 2.3 = Draw
Incorrect!
NL vs JP: 2 - 2 = Draw
NL vs JP: 3.3 - 0.1 = NL
Incorrect!
SE vs TN: 5 - 1 = SE
SE vs TN: 0.7 - 0.7 = Draw
Incorrect!
BE vs EG: 1 - 1 = Draw
BE vs EG: 0.9 - 0.4 = Draw
Correct!
IR vs NZ: 2 - 2 = Draw
IR vs NZ: 1.8 - 0.7 = IR
Incorrect!
ES vs CV: 0 - 0 = Draw
ES vs CV: 4.9 - 1.5 = ES
Incorrect!
SA vs UY: 1 - 1 = Draw
SA vs UY: 0.3 - 10.0 = UY
Incorrect!
FR vs SN: 3 - 1 = FR
FR vs SN: 4.9 - 0.0 = FR
Correct!
IQ vs NO: 1 - 4 =

In [5]:
print(f"Accuracy on the result: {correct_winners / len(df_results)}")

# how accurate is the prediction of the goals? MAE
combined_df = df_results.merge(local_predictions, on=['team_1', 'team_2'], how='inner')
combined_df['mae_1'] = abs(combined_df['goals_1_x'] - combined_df['goals_1_y'])
combined_df['mae_2'] = abs(combined_df['goals_2_x'] - combined_df['goals_2_y'])
combined_df['mae'] = (combined_df['mae_1'] + combined_df['mae_2']) / 2
print(f"MAE on the prediction on the goals: {combined_df['mae'].mean()}")

Accuracy on the result: 0.375
MAE on the prediction on the goals: 2.0088668364162086
